# 02 — Clean, join, and describe the country-month panel

Builds `data/processed/panel_base.parquet`: one row per **Gleditsch–Ward state × month, Jan 1989 – Dec 2025**, with UCDP GED conflict deaths, V-Dem covariates, and a neighbour-conflict column. Then describes it.

**Inputs** (`data/raw/`, produced by `01_download.ipynb`): GED 26.1, Gleditsch–Ward state list, V-Dem v16 Core, COW Direct Contiguity 3.2.

**Outputs** (`data/processed/`): `panel_base.parquet` (the panel), `contiguity_gw.parquet` (GW-coded neighbour pairs by year).

**What this notebook does *not* do:** build model features (lags, rolling means). Those come in the next step, so that "what the data is" stays separate from "what the model sees" and leakage is easy to audit. The one exception is `nbr_sb_best`, which is **same-month and unlagged** — it describes the month and must be lagged before any model sees it.

### Conventions (settled — see `Planning/Final Project Guide.md`)
| Decision | Rule |
|---|---|
| Universe | Gleditsch–Ward main list; microstates excluded (none appear in GED) |
| GW end date 31 Dec 2017 | Censoring date of the list, not an exit → extended to Dec 2025 |
| Multi-month GED events | Deaths spread evenly over the days covered, then summed by month |
| Violence types | `sb` state-based (target), `ns` non-state, `os` one-sided |
| Fatalities | `best` primary; `low` / `high` kept for robustness |
| Annual covariates | **y−1 rule**: every month of year *y* gets the year *y−1* value |
| COW contiguity | Ends 2016; 2016 borders carried forward to 2025 (no changes since) |

## Setup

In [ ]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root == root.parent:
        raise FileNotFoundError("Open this notebook from inside the DACSS_ML_FP repo.")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.paths import RAW, PROCESSED
from src.data import panel as P

PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140, "display.max_columns", 30)

# Input files — change here if a filename differs.
GED_ZIP    = RAW / "ged261-csv.zip"
GW_FILE    = RAW / "gw_iisystem.dat"
VDEM_FILE  = RAW / "V-Dem-CY-Core-v16.csv"
CONTIG_ZIP = RAW / "DirectContiguity320.zip"

# Chart style: thin marks, recessive grid, one colour per violence type (fixed order).
COLORS = {"sb": "#2a78d6", "ns": "#eb6834", "os": "#1baf7a"}
LABELS = {"sb": "State-based", "ns": "Non-state", "os": "One-sided"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8a85", "axes.labelcolor": "#3d3d3a", "xtick.color": "#5f5e5a",
    "ytick.color": "#5f5e5a", "axes.grid": True, "grid.color": "#e5e4df", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "lines.linewidth": 1.6, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
})

# Spreading multi-month events creates fractional deaths (a 5-death event over a
# year is ~0.4 per month). For yes/no descriptives, a "conflict month" needs >= 1 death.
CONFLICT_MIN = 1.0

## 1. The country-month grid

Every month in which a state exists on at least one day gets a row. Peaceful months are absent from GED, so this grid is what makes them visible as zeros — and keeps zeros out of months when a state did not exist (South Sudan before July 2011, post-Soviet states before 1991).

In [ ]:
gw = P.load_gw(GW_FILE)
grid = P.build_grid(gw)

in_panel = gw[gw["gwno"].isin(grid["gwno"])]
print(f"GW spells in file:              {len(gw):,}")
print(f"States in the panel:            {grid['gwno'].nunique():,}")
print(f"Country-months:                 {len(grid):,}")
print(f"Spells censored at 31 Dec 2017: {int(in_panel['censored_2017'].sum())} (extended to Dec 2025)")

# Real exits and entries inside the panel window
exits = in_panel[(~in_panel["censored_2017"]) & (in_panel["end"] >= "1989-01-01")]
entries = in_panel[in_panel["start"] >= "1989-01-01"]
print(f"\nExits during 1989–2025: {len(exits)}")
display(exits[["gwno", "name", "start", "end"]])
print(f"Entries during 1989–2025: {len(entries)}")
display(entries[["gwno", "name", "start"]].reset_index(drop=True))

## 2. GED events → country-month deaths

Steps: load events → confirm every event falls inside its state's lifespan → spread multi-month events across the days they cover → sum by country, month, and violence type → attach to the grid and fill the gaps with zero.

**Why spread?** Multi-month events are under 2% of events but carry about a quarter of all deaths, including single events coded across a whole year (Ethiopia–TPLF 2021: 113k deaths, Jan 1 – Dec 31). Assigning them to their start month would put a year of deaths into one January.

In [ ]:
ev = P.load_ged(GED_ZIP)
print(f"Events: {len(ev):,}   years {ev['year'].min()}–{ev['year'].max()}")
display(ev["type_of_violence"].map(P.VIOLENCE_TYPES).map(LABELS).value_counts().rename("events").to_frame())

outside = P.events_outside_lifespan(ev, gw)
print(f"Events outside their state's GW lifespan: {len(outside)}")
assert outside.empty, "some GED events fall outside their state's existence — inspect `outside`"


In [ ]:
pieces = P.spread_events(ev)

# Spreading must not create or destroy deaths.
for c in P.FATALITY_COLS:
    assert np.isclose(pieces[c].sum(), ev[c].sum()), c

m_start = ev["date_start"].dt.to_period("M")
multi = ev[m_start != ev["date_end"].dt.to_period("M")].copy()
multi["days"] = (multi["date_end"] - multi["date_start"]).dt.days + 1
print(f"Multi-month events:   {len(multi):,} ({len(multi)/len(ev):.1%} of events)")
print(f"Deaths they carry:    {multi['best'].sum():,.0f} ({multi['best'].sum()/ev['best'].sum():.1%} of all 'best' deaths)")
print(f"Median span:          {multi['days'].median():.0f} days;  events spanning ≥ 1 year: {(multi['days'] >= 365).sum()}")
print("\nLargest multi-month events:")
display(multi.nlargest(10, "best")[["country", "type_of_violence", "best", "date_start", "date_end", "days"]]
        .reset_index(drop=True))

In [ ]:
ged_m = P.aggregate_ged(pieces)
death_event_cols = [c for c in ged_m.columns if c not in ("gwno", "month")]

panel = grid.merge(ged_m, on=["gwno", "month"], how="left", validate="one_to_one")
panel[death_event_cols] = panel[death_event_cols].fillna(0)

# Every GED country-month must land on the grid, and every death must arrive.
assert len(ged_m.merge(grid, on=["gwno", "month"])) == len(ged_m), "GED months outside the grid"
for t, p in P.VIOLENCE_TYPES.items():
    raw = ev.loc[ev["type_of_violence"] == t, "best"].sum()
    assert np.isclose(panel[f"{p}_best"].sum(), raw), p
print(f"Panel: {panel.shape[0]:,} rows × {panel.shape[1]} columns — all GED deaths accounted for")
print("Note: {sb,ns,os}_events counts events *active* in the month, so a 3-month event counts in each month.")

## 3. V-Dem covariates (y−1 rule)

Joined on V-Dem's stable `country_id`, not the COW code, because V-Dem keeps one id across code changes (Germany 260→255, Yemen 678→679, Czechia 315→316). One manual mapping: **Serbia** is GW 340 from June 2006 but V-Dem files it under COW 345 throughout.

| Variable | Meaning (0–1 unless noted) |
|---|---|
| `v2x_polyarchy` | Electoral democracy index |
| `v2x_libdem` | Liberal democracy index |
| `v2x_civlib` | Civil liberties index |
| `v2x_clphy` | Physical violence index (higher = more freedom from political killings/torture) |
| `v2x_rule` | Rule of law index |
| `v2x_corr` | Political corruption index (higher = more corrupt) |
| `v2x_regime` | Regimes of the World: 0 closed autocracy, 1 electoral autocracy, 2 electoral democracy, 3 liberal democracy |

**Interpretation caution.** These values change once a year, so they explain differences *between* countries and slow drift, not month-to-month swings. Effective information is ~6,300 country-years, not 75,700 rows.

In [ ]:
vdem = P.load_vdem(VDEM_FILE)
print(f"V-Dem rows: {len(vdem):,}   years {vdem['year'].min()}–{vdem['year'].max()}")

xwalk = P.gw_to_vdem_map(gw, vdem)
no_vdem = xwalk[xwalk["vdem_id"].isna() & xwalk["gwno"].isin(panel["gwno"])]
print("\nPanel states with no V-Dem series:")
display(no_vdem.merge(gw[["gwno", "name"]].drop_duplicates("gwno")))

panel = P.join_vdem_lagged(panel, vdem, xwalk)

miss = (panel[panel["vdem_missing"]]
        .groupby(["gwno", "country"])["month"].agg(rows="size", first="min", last="max")
        .reset_index())
print(f"Rows without V-Dem values: {panel['vdem_missing'].sum():,} ({panel['vdem_missing'].mean():.2%})")
display(miss)
print("Missing for whole series = state not scored by V-Dem. "
      "Missing for a few months at the start = state born that year, so no y−1 value exists.")
print("Left as NaN here; the fill-or-drop rule is a modelling decision.")

## 4. COW contiguity → neighbour conflict

COW codes are converted to GW (Germany 255→260, Yemen 679→678, Serbia 345→340 from 2006). 2016 borders are carried forward to 2025. **Neighbour = land or river border** (`conttype` 1); the water-distance types 2–5 are kept in `contiguity_gw.parquet` for a robustness check.

`nbr_sb_best` = state-based deaths summed over a country's land neighbours **in the same month**. Unlagged — describes the month. `n_land_nbrs` counts neighbours that exist that month (0 for island states).

In [ ]:
dyads = P.load_contiguity_gw(CONTIG_ZIP, gw)
n_before = len(dyads)
dyads = P.backfill_early_years(dyads, panel)
print(f"Backfilled {len(dyads) - n_before} dyad-years for states born before COW codes them (Macedonia 1991-92)")
print(f"GW-coded directed dyad-years: {len(dyads):,}   years {dyads['year'].min()}–{dyads['year'].max()}")
display(dyads["conttype"].value_counts().sort_index().rename("dyad-years").to_frame())

panel["nbr_sb_best"] = P.neighbour_sum(panel, dyads, "sb_best", max_conttype=1)
panel["_one"] = 1
panel["n_land_nbrs"] = P.neighbour_sum(panel, dyads, "_one", max_conttype=1).astype(int)
panel = panel.drop(columns="_one")

print(f"Country-months with no land neighbour: {(panel['n_land_nbrs'] == 0).mean():.1%}")
never = panel.groupby("country")["n_land_nbrs"].max().eq(0)
print(f"States with no land neighbour in ANY month ({never.sum()}): {', '.join(sorted(never[never].index))}")
sometimes = panel[panel["n_land_nbrs"].eq(0) & ~panel["country"].isin(never[never].index)]
print(f"Zero-neighbour months in states that do have neighbours at other times: {len(sometimes)}")
print("Note: COW codes 1991 as a transition year, so Russia's neighbour count peaks at 20 in Dec 1991 "
      "(USSR-era and successor-state borders both listed). One country-month; left as is.")

## 5. Invariant checks

If any of these fail, nothing downstream can be trusted. They are assertions, so a failure stops the notebook.

In [ ]:
checks = {}
checks["one row per (gwno, month)"] = not panel.duplicated(["gwno", "month"]).any()
checks["row count matches grid"] = len(panel) == len(grid)

# No internal gaps: within each state, months are consecutive.
idx = panel["month"].dt.year * 12 + panel["month"].dt.month   # integer month index (avoids Period arithmetic)
gaps = idx.groupby(panel["gwno"]).apply(lambda s: (s.sort_values().diff().dropna() != 1).sum())
checks["no internal month gaps"] = int(gaps.sum()) == 0

checks["no negative deaths/events"] = (panel[death_event_cols] >= 0).all().all()
checks["months within 1989-01 … 2025-12"] = panel["month"].between(P.PANEL_START, P.PANEL_END).all()
vd = panel[[v for v in P.VDEM_VARS if v != "v2x_regime"]].stack().dropna()  # pandas 3 stack keeps NaN
checks["V-Dem indices within [0, 1]"] = vd.between(0, 1).all()
checks["v2x_regime in {0,1,2,3}"] = panel["v2x_regime"].dropna().isin([0, 1, 2, 3]).all()

display(pd.Series(checks, name="pass").to_frame())
assert all(checks.values()), "an invariant failed"

panel = panel.sort_values(["gwno", "month"], ignore_index=True)

## 6. Descriptive statistics

### 6.1 Panel structure

In [ ]:
lengths = panel.groupby("gwno").size()
ever_sb = panel.groupby("gwno")["sb_best"].sum() > 0
structure = pd.Series({
    "states": panel["gwno"].nunique(),
    "months": panel["month"].nunique(),
    "country-months (rows)": len(panel),
    "full-length series (444 months)": int((lengths == 444).sum()),
    "shortest series (months)": int(lengths.min()),
    "states with any state-based deaths": int(ever_sb.sum()),
    "conflict months (≥ 1 state-based death)": int((panel["sb_best"] >= CONFLICT_MIN).sum()),
    "share of country-months with < 1 state-based death": f"{(panel['sb_best'] < CONFLICT_MIN).mean():.1%}",
    "months with fractional deaths (0 < deaths < 1)": int(((panel["sb_best"] > 0) & (panel["sb_best"] < CONFLICT_MIN)).sum()),
}, name="value")
display(structure.to_frame())
print("Shortest series:")
display(lengths.nsmallest(8).rename("months").to_frame()
        .join(panel.drop_duplicates("gwno").set_index("gwno")["country"]))

### 6.2 The target: state-based deaths

About 85% of country-months have no state-based deaths. A model can score well by predicting zero almost everywhere, so evaluation must also report results on **conflict months** and **onsets**. Among conflict months the distribution is extremely right-skewed — hence the log(1 + deaths) target. (Conflict month = ≥ 1 death; the ~200 months with a fraction of a death come from spreading long events and are kept in the target as they are.)

In [ ]:
y = panel["sb_best"]
nz = y[y >= CONFLICT_MIN]
q = nz.quantile([.25, .5, .75, .9, .99])
summary = pd.Series({
    "months with < 1 death": f"{(y < CONFLICT_MIN).mean():.1%}",
    "conflict months (≥ 1)": f"{len(nz):,}",
    "median | conflict month": f"{q[.5]:,.1f}",
    "90th pct | conflict month": f"{q[.9]:,.0f}",
    "99th pct | conflict month": f"{q[.99]:,.0f}",
    "max": f"{y.max():,.0f}",
    "skewness (raw)": f"{y.skew():.1f}",
    "skewness log1p": f"{np.log1p(y).skew():.2f}",
}, name="state-based deaths")
display(summary.to_frame())

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(np.log1p(nz), bins=50, color=COLORS["sb"], edgecolor="white", linewidth=0.6)
ticks = [1, 10, 100, 1_000, 10_000]
ax.set_xticks(np.log1p(ticks), [f"{t:,}" for t in ticks])
ax.set_xlabel("State-based deaths in the month (log scale)")
ax.set_ylabel("Country-months")
ax.set_title(f"Conflict months only (n = {len(nz):,}); a further {(y < CONFLICT_MIN).sum():,} have < 1 death")
plt.tight_layout(); plt.show()

In [ ]:
print("Largest country-months (state-based, best):")
display(panel.nlargest(15, "sb_best")[["country", "month", "sb_best", "sb_events", "os_best", "nbr_sb_best"]]
        .reset_index(drop=True).round({"sb_best": 0, "os_best": 0, "nbr_sb_best": 0}))

### 6.3 Over time

In [ ]:
yearly = panel.assign(year=panel["month"].dt.year).groupby("year")
zero_share = yearly["sb_best"].apply(lambda s: (s < CONFLICT_MIN).mean())
active = yearly.apply(lambda d: d.loc[d["sb_best"] >= CONFLICT_MIN, "gwno"].nunique())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
axes[0].plot(zero_share.index, zero_share.to_numpy() * 100, color=COLORS["sb"])
axes[0].set_title("Country-months without conflict (%)")
axes[0].set_ylim(70, 100)
axes[1].plot(active.index, active.to_numpy(), color=COLORS["sb"])
axes[1].set_title("States with any conflict month that year")
axes[1].set_ylim(0, None)
plt.tight_layout(); plt.show()

In [ ]:
monthly = panel.groupby("month")[[f"{p}_best" for p in P.VIOLENCE_TYPES.values()]].sum()
x = monthly.index.to_timestamp()

# Small multiples, each with its own y-scale: the three types differ by an order of magnitude.
fig, axes = plt.subplots(3, 1, figsize=(10, 6.6), sharex=True)
for ax, p in zip(axes, P.VIOLENCE_TYPES.values()):
    ax.plot(x, monthly[f"{p}_best"] / 1000, color=COLORS[p], linewidth=1.2)
    ax.set_title(f"{LABELS[p]} deaths per month, worldwide (thousands)")
    ax.set_ylim(0, None)
axes[-1].set_xlabel("Month")
plt.tight_layout(); plt.show()
print("Flat plateaus (e.g. Ethiopia 2021, Syria 2013–14) are year-long GED events spread evenly across months.")

### 6.4 Persistence — how hard is the baseline?

If this month's deaths almost fully determine next month's, the persistence baseline ("next month = this month") will be very hard to beat. These numbers preview that, within countries only (no month is paired across two countries).

In [ ]:
g = panel.groupby("gwno")["sb_best"]
cur = panel["sb_best"]
nxt = g.shift(-1)
ok = nxt.notna()

state_now = np.where(cur[ok] >= CONFLICT_MIN, "conflict", "no conflict")
state_next = np.where(nxt[ok] >= CONFLICT_MIN, "conflict", "no conflict")
trans = pd.crosstab(pd.Series(state_now, name="this month"), pd.Series(state_next, name="next month"),
                    normalize="index")
display((trans * 100).round(1).astype(str) + "%")

r = np.corrcoef(np.log1p(cur[ok]), np.log1p(nxt[ok]))[0, 1]
nzmask = (cur[ok] >= CONFLICT_MIN) & (nxt[ok] >= CONFLICT_MIN)
r_nz = np.corrcoef(np.log1p(cur[ok][nzmask]), np.log1p(nxt[ok][nzmask]))[0, 1]
print(f"Correlation of log1p(deaths) this month vs next month: {r:.3f}")
print(f"  … among pairs that are conflict months in both:        {r_nz:.3f}")

### 6.5 Onset groundwork

The onset definition is still open: **an onset is a month with ≥ *k* state-based deaths after ≥ *m* consecutive months below *k*.** UCDP's 25-death threshold is per dyad-*year*, so it cannot be borrowed for months.

The table counts onsets for a grid of *k* and *m*. Only observed months count toward *m*, so a state's first *m* months can never be onsets (we cannot see their history). Pick a definition that leaves enough onsets to evaluate on — a few hundred at minimum — and test neighbouring values for robustness.

In [ ]:
def quiet_run_before(s: pd.Series, k: float) -> pd.Series:
    """For each month, the number of consecutive prior months with deaths < k."""
    quiet = (s < k).astype(int).to_numpy()
    run = np.zeros(len(quiet), dtype=int)
    for i in range(1, len(quiet)):
        run[i] = run[i - 1] + 1 if quiet[i - 1] else 0
    return pd.Series(run, index=s.index)

ks = [1, 5, 10, 25]
ms = [1, 3, 6, 12, 24]
rows = {}
for k in ks:
    runs = panel.groupby("gwno", group_keys=False)["sb_best"].apply(quiet_run_before, k=k)
    hit = panel["sb_best"] >= k
    rows[f"k = {k}"] = {f"m = {m}": int((hit & (runs >= m)).sum()) for m in ms}
onsets = pd.DataFrame(rows).T
print("Number of onset months, 1989–2025:")
display(onsets)
print("For scale: the locked test period (2021–2025) is ~13.5% of all country-months.")

### 6.6 Data quality

In [ ]:
prec = ev["date_prec"].value_counts().sort_index()
prec_lab = {1: "exact day", 2: "2–6 days", 3: "week", 4: "month", 5: "> month (e.g. year)"}
display(pd.DataFrame({"events": prec, "deaths (best)": ev.groupby("date_prec")["best"].sum().round(0)})
        .rename(index=prec_lab).rename_axis("date precision"))

# GED's own bounds are not always ordered: a source quirk, reported not fixed.
bad_ev = ev[(ev["low"] > ev["best"]) | (ev["best"] > ev["high"])]
print(f"GED events where low ≤ best ≤ high does not hold: {len(bad_ev):,} ({len(bad_ev)/len(ev):.1%}); "
      f"{(bad_ev['year'] < 2000).mean():.0%} of them before 2000")
display(bad_ev.groupby("type_of_violence")[["low", "best", "high"]].sum()
        .rename(index=lambda t: LABELS[P.VIOLENCE_TYPES[t]]).round(0))
print("Consequence: low/high are usable only as a rough robustness band; 'best' is the target.")

ratios = pd.DataFrame({p: {"low / best": panel[f"{p}_low"].sum() / panel[f"{p}_best"].sum(),
                           "high / best": panel[f"{p}_high"].sum() / panel[f"{p}_best"].sum()}
                       for p in P.VIOLENCE_TYPES.values()}).rename(columns=LABELS).round(3)
print("Uncertainty band around 'best', total deaths:")
display(ratios)

### 6.7 Covariates

Same-row correlations below are **descriptive only** (and V-Dem is already lagged a year). They say nothing about prediction or causation.

In [ ]:
cov_cols = P.VDEM_VARS + ["nbr_sb_best", "n_land_nbrs"]
display(panel[cov_cols].describe().T[["count", "mean", "std", "min", "50%", "max"]].round(3))

regime = panel["v2x_regime"].map({0: "closed autocracy", 1: "electoral autocracy",
                                  2: "electoral democracy", 3: "liberal democracy"})
display(pd.DataFrame({
    "country-months": regime.value_counts(),
    "share that are conflict months": panel.groupby(regime)["sb_best"].apply(lambda s: (s >= CONFLICT_MIN).mean()).round(3),
}))

corr = panel.assign(log_sb=np.log1p(panel["sb_best"]), log_nbr=np.log1p(panel["nbr_sb_best"]))[
    ["log_sb", "log_nbr"] + P.VDEM_VARS].corr(method="spearman")["log_sb"].drop("log_sb")
print("Spearman correlation with log1p(state-based deaths), same row:")
display(corr.round(3).to_frame("ρ"))

## 7. Save

`month` is stored as a timestamp (first day of the month) because Parquet has no native monthly-period type; convert back with `.dt.to_period("M")`.

In [ ]:
out = panel.copy()
out["month"] = out["month"].dt.to_timestamp()
out_path = PROCESSED / "panel_base.parquet"
out.to_parquet(out_path, index=False)
dyads.to_parquet(PROCESSED / "contiguity_gw.parquet", index=False)

print(f"Saved {out_path.relative_to(root)}: {out.shape[0]:,} rows × {out.shape[1]} columns")
print(pd.Series(out.dtypes.astype(str), name="dtype").to_frame().T.to_string())